# SAVI: example

Runs the whole analysis on the output of the data preparation (`SAVI_input.rds`, written by `SAVI_data_prep_example.ipynb`). R kernel.

| Step | What | File |
|---|---|---|
| 0 | Settings and libraries | |
| 1 | Load `SAVI_input.rds` and the pseudotime | |
| 2 | **Choose `sp_num` / `time_num` by simulation** (optional) | `SAVI_tuning.R` |
| 3 | Quick test on 2 genes (`fusing_bench()`) | `SAVI.R` |
| 4 | Full SAVI run | `SAVI.R` |
| 5 | Look at the results | |
| 6 | **Baselines** (global lasso, Moran's R; optional eSpatial, Signac, Cicero) | `SAVI_evaluation.R` |
| 7 | **Evaluation** (gene-CRE validation, cross-modality reproducibility; both optional) | `SAVI_evaluation.R` |
| 8 | **Split-half reproducibility** (optional; needs no second modality) | `SAVI_evaluation.R` |

Needed: `SAVI.R`, `SAVI_tuning.R`, `SAVI_evaluation.R`, `SAVI_input.rds`, one pseudotime value per spatial cell (a csv with column `x`; the same pseudotime is used for every gene, e.g. the ATAC-based pseudotime), and the R packages `furrr`, `future`, `glmnet`, `dplyr` and `myFuser` (provides `fusedLassoProximal`). Step 7 additionally uses `pROC`, `PRROC`, `ggplot2`, `GenomicRanges`, `Signac` when a gene-CRE table is given.
This notebook does **not** need `SAVI_data_prep.R`. `SAVI_tuning.R` and `SAVI_evaluation.R` are independent of `SAVI.R` and of each other.

## 0. Settings and libraries (edit the paths)

In [ ]:
# ---- paths and settings: edit these ----
work_dir    <- "/storage10/shuyang/MISAR-seq-main"
input_rds   <- file.path(work_dir, "SAVI_input.rds")
pseudotime_file <- file.path(work_dir, "E15_pseudoT.csv")   # csv, column x = pseudotime of each spatial cell
n_workers   <- 25

# grouping parameters. Set both to NULL to take them from the tuning step (Step 2),
# or give numbers to skip it.
sp_num   <- 4                         # number of spatial bins along x (and y)
time_num <- 3                         # number of pseudotime bins

# ---- optional inputs for Steps 6-7 (leave as NULL / FALSE when not available) ----
gene_cre_file <- NULL                    # table with columns Gene, Coordinate1, Coordinate2 (chr_start_end)
espatial_file <- NULL                    # .Rdata holding the eSpatial tables GPTab and enh.pattern
second_modality_result <- NULL           # e.g. SAVI result of H3K27ac: file.path(work_dir, "SAVI_result_h3k27ac.Rdata")
run_split_half <- TRUE                  # Step 8: split-half reproducibility (runs SAVI twice on half of the cells)
n_splits       <- 1
run_signac_cicero <- TRUE               # needs the spatial Seurat object, fragments, annotation
fragment_path <- file.path(work_dir,"Data/E15_5-S1_filtered_fragments.tsv.bgz")
sp_data = readRDS(file.path(work_dir,"E15_5-S1_SpatialSeurat_raw.rds"))
load('sample_gene.Rdata')

library(future)
library(furrr)
library(glmnet)
library(dplyr)
library(myFuser)                         # fusedLassoProximal()

# SAVI.R defines fusing_bench() and its helpers.
source("SAVI.R")
source("SAVI_tuning.R")                  # Module 1: sp_num / time_num selection
source("SAVI_evaluation.R")              # Module 2: baselines and evaluation

In [ ]:
library(EnsDb.Mmusculus.v79)
gene_annotation <- GetGRangesFromEnsDb(
  ensdb = EnsDb.Mmusculus.v79
)

# Convert chromosome names to UCSC style:
# 1 -> chr1
# 2 -> chr2
# ...
seqlevelsStyle(gene_annotation) <- "UCSC"

# Set genome
genome(gene_annotation) <- "mm10"

## 1. Load the SAVI input and the pseudotime

`pseudoT` is ONE numeric vector with the pseudotime of every spatial cell, in the same cell order as `prep$spatial_coord`. It is shared by all genes, so all genes use the same groups.

In [ ]:
prep <- readRDS(input_rds)
prep$use_genes <- gene
pseudoT <- read.csv(pseudotime_file)$x
genes   <- prep$use_genes
stopifnot(length(genes) > 0)

# the pseudotime must have one value per spatial cell
n_cells <- nrow(prep$spatial_coord)
stopifnot(length(pseudoT) == n_cells)

cat("cells:", n_cells, "| genes analysed:", length(genes), "\n")

## 2. Choose `sp_num` and `time_num` by simulation (Module 1)

`SAVI_tune_grid()` takes the **first gene** and its candidate peaks (imputed ATAC = real predictors), declares the **first 5 peaks the true enhancers** and simulates the gene's expression from them (enhancer effects vary smoothly in space and pseudotime). It then fits SAVI's fused lasso for every `sp_num` x `time_num` in the grid, scores the estimates against the known truth (AUROC, AUPR, precision, recall, F1, averaged over groups) and recommends the best pair.

First look at the groupings the grid would produce on the real data (instant, no fitting); settings that keep few cells or make tiny groups can be removed from the grid.

In [ ]:
sp_grid   <- c(3, 5, 7, 10)
time_grid <- c(1, 3, 5, 7)

SAVI_grouping_summary(prep$spatial_coord, pseudoT, sp_grid, time_grid)

Run the search. Start with `n_rep = 1` and a small grid to check the timing: each setting runs a grouped 5-fold CV over the lambda x gamma grid. Set `n_cores` for parallel settings.

In [ ]:
run_tuning <- is.null(sp_num) || is.null(time_num)

if (run_tuning) {
  tuning <- SAVI_tune_grid(
    imputation = prep$imputed, spatial_coord = prep$spatial_coord,
    pseudoT = pseudoT, use_genes = genes, use_peaks = prep$use_peaks[genes],
    gene = 1, true_peaks = 1:5,
    sp_nums = sp_grid, time_nums = time_grid,
    n_rep = 2, criterion = "composite", min_keep_frac = 0.8,
    n_cores = n_workers
  )
  saveRDS(tuning, file.path(work_dir, "SAVI_tuning.rds"))
  print(tuning$summary)
  print(plot_tuning_grid(tuning))
  sp_num   <- tuning$recommended$sp_num
  time_num <- tuning$recommended$time_num
}
cat("using sp_num =", sp_num, " time_num =", time_num, "\n")

# fusing_bench() takes bin widths, not bin counts
sp_interval   <- diff(range(prep$spatial_coord[, 1])) / sp_num
time_interval <- diff(range(pseudoT)) / time_num

## 3. Quick test on 2 genes

SAVI is slow (grouped cross-validation over a lambda x gamma grid and 1000 permutations per gene), so first check that everything runs on a few genes.

In [ ]:
plan(multisession, workers = n_workers)

test_genes <- genes[1:2]
res_test <- fusing_bench(
  sp_interval = sp_interval, time_interval = time_interval,
  spatial_coord = prep$spatial_coord,
  pseudoT = pseudoT,
  use_peaks = prep$use_peaks[test_genes],
  use_genes = test_genes,
  imputation = prep$imputed,
  cell_types = prep$cell_types
)
names(res_test$coef)
names(res_test$coef[[1]])

## 4. Full run

In [ ]:
res <- fusing_bench(
  sp_interval = sp_interval, time_interval = time_interval,
  spatial_coord = prep$spatial_coord,
  pseudoT = pseudoT,
  use_peaks = prep$use_peaks[genes],
  use_genes = genes,
  imputation = prep$imputed,
  cell_types = prep$cell_types
)
save(res, file = file.path(work_dir, sprintf("SAVI_result_sp%d_t%d.Rdata", sp_num, time_num)))
plan(sequential)

## 5. Results

`res$coef[[gene]]` holds, for each gene:

* `fusion`: peak x group coefficients of the final fused lasso (tuned lambda and gamma)
* `fusion_old`: fused lasso with fixed lambda = gamma = 0.001
* `lasso`: within-group elastic-net coefficients (baseline)
* `marginal`, `marginal_p`: within-group Spearman correlation of each peak with the gene, and its permutation p-value (peaks x groups)
* `fusion_cv`: tuned lambda / gamma and the cross-validation grid

The spatiotemporal grouping is shared by all genes and stored once at the top level: `res$groups`, `res$group_all`, `res$group_count`.

In [ ]:
g <- names(res$coef)[1]
r <- res$coef[[g]]

str(r, max.level = 1)
head(res$group_count)                                    # groups (x-y-t bins) and their cell counts

# peaks with the strongest fused-lasso coefficients across groups
head(sort(rowMeans(abs(r$fusion)), decreasing = TRUE), 10)

# permutation p-values of the marginal correlation, first groups
head(r$marginal_p[, 1:3])

## 6. Baselines (Module 2)

`SAVI_baselines()` adds the comparison methods. Always available (computed from the imputed data): **global lasso** (`Lasso(g)`) and **Moran's R**. The within-group lasso (`Lasso(w)`) and the marginal correlation (`Marginal`) are already in `res`.

Optional (skipped when their input is not given): **eSpatial** (needs the eSpatial tables `GPTab` and `enh.pattern`, produced with the eSpatial code), **Signac links** and **Cicero** (need the spatial Seurat object, its fragment file and a gene annotation, via `build_link_object()`).

Moran's R bandwidth defaults to range(x)/20; the original analysis script used `max(x) - min(y)` / 20 (a typo mixing the two coordinates). Pass `moran_bandwidth = max(x) - min(y)` / 20 to reproduce the earlier numbers exactly.

In [ ]:
load('SAVI_result_sp4_t3.Rdata')

In [ ]:
res_genes <- names(res$coef)

eSpatial_input <- NULL
if (!is.null(espatial_file) && file.exists(espatial_file)) {
  load(espatial_file)                    # provides GPTab and enh.pattern
  eSpatial_input <- list(GPTab = GPTab, enh_pattern = enh.pattern)
}

link_object <- NULL
if (run_signac_cicero) {
  # sp_data: spatial Seurat object with an SCT assay; fragments: path to the fragment file;
  # annotation: gene annotation (e.g. Annotation(sp_data[["peaks"]])).  Edit before enabling.
  link_object <- build_link_object(sp_data, res_genes, prep$use_peaks[res_genes],
                                   fragments = fragment_path, annotation = gene_annotation)
}

methods <- c("global_lasso", "moran",
             if (!is.null(eSpatial_input)) "eSpatial",
             if (!is.null(link_object)) c("signac", "cicero"))

baselines <- SAVI_baselines(prep$imputed, res_genes, prep$use_peaks[res_genes], prep$spatial_coord,
                            methods = methods, eSpatial_input = eSpatial_input,
                            link_object = link_object, n_cores = n_workers)
names(baselines)
#saveRDS(baselines, file.path(work_dir, "SAVI_baselines.rds"))

In [ ]:
baselines = readRDS('SAVI_baselines.rds')

## 7. Evaluation (Module 2)

`SAVI_evaluate()` runs whatever its inputs allow and prints what it skipped:

* **Gene-CRE validation** (needs `gene_cre`: known gene-CRE pairs with columns `Gene`, `Coordinate1`, `Coordinate2`): per-gene AUROC / AUPR of every method and precision / recall at the top-k peaks.
* **Cross-modality reproducibility** (needs a SAVI result of a second modality, e.g. H3K27ac, in `res_h`): Spearman correlation and Jaccard index of the results between the two modalities.

Without H3K27ac and without a gene-CRE table nothing in this step can run; the baselines of Step 6 are still saved.

In [ ]:
gene_cre <- NULL
if (!is.null(gene_cre_file) && file.exists(gene_cre_file)) {
  gene_cre <- unique(read.table(gene_cre_file, header = TRUE)[, c("Coordinate1", "Gene", "Coordinate2")])
}

res_h <- NULL; baselines_h <- list()
if (!is.null(second_modality_result) && file.exists(second_modality_result)) {
  load(second_modality_result)           # must provide `res_h` and (optionally) `baselines_h`
}

evaluation <- SAVI_evaluate(
  res, res_genes, prep$use_peaks[res_genes], baselines = baselines,
  gene_cre = gene_cre, res_h = res_h, baselines_h = baselines_h, n_cores = n_workers
)
names(evaluation)

if (!is.null(evaluation$plots)) lapply(evaluation$plots, print)
if (!is.null(evaluation$validation_metrics)) aggregate(cbind(auroc, aupr) ~ Method, evaluation$validation_metrics, mean)
saveRDS(evaluation, file.path(work_dir, "SAVI_evaluation.rds"))

## 8. Split-half reproducibility (Module 2, optional)

The cells are split at random into two equal halves, SAVI and the baselines are run on each half with the same settings, and the two results are compared (Spearman correlation and Jaccard index of the coefficients, per gene and method). This needs no second modality. It runs SAVI twice, so enable it with `run_split_half <- TRUE` in the settings.

Signac / Cicero are included when `link_object` (Step 6) exists; `spatial_coord` must then have the cell names as rownames.

In [ ]:
link_object = baselines$link_object
sp_interval   <- diff(range(prep$spatial_coord[, 1])) / sp_num
time_interval <- diff(range(pseudoT)) / time_num

In [ ]:
if (run_split_half) {
  plan(multisession, workers = n_workers)
  split_eval <- SAVI_split_reproducibility(
    imputation = prep$imputed, spatial_coord = prep$spatial_coord, pseudoT = pseudoT,
    use_genes = genes, use_peaks = prep$use_peaks[genes], cell_types = prep$cell_types,
    sp_interval = sp_interval, time_interval = time_interval,
    methods = c("global_lasso", "moran", if (!is.null(link_object)) c("signac", "cicero")),
    link_object = link_object, n_splits = n_splits, n_cores = n_workers
  )
  plan(sequential)
  print(split_eval$summary)
  print(split_eval$plot)
  saveRDS(split_eval, file.path(work_dir, "SAVI_split_half.rds"))
}